In [ ]:
import cv2
from deepface import DeepFace
import numpy as np
from typing import Sequence, List, Tuple
import os

In [ ]:
# https://github.com/opencv/opencv/blob/master/data/haarcascades/

In [ ]:
def smooth(x: Sequence[int], y: Sequence[int], sigma: float = 1) -> Tuple[List[int], List[int]]:
    return list(x), list(y) # TODO
    # # a simple moving average
    # from scipy.ndimage import gaussian_filter1d
    
    # x_array = np.array(x, dtype=float)
    # y_array = np.array(y, dtype=float)
    
    # x_smooth = gaussian_filter1d(x_array, sigma=sigma)
    # y_smooth = gaussian_filter1d(y_array, sigma=sigma)
    
    # return x_smooth.astype(int).tolist(), y_smooth.astype(int).tolist()

def process_video_with_smoothed_crop(
    input_video_path: str,
    output_video_path: str,
    crop_size: Tuple[int, int] = (224, 224),  # Output crop size
    target_fps: int | None = None,  # If None, uses original FPS
    detector_backend: str = 'opencv',
    smoothing_sigma: float = 1.0,
    buffer_size: int = 100,  # Buffer for collecting center positions
    show_preview: bool = False  # Show processing preview
):
    """
    Process video with smoothed face crop.
    
    Args:
        input_video_path: Path to input MP4 video
        output_video_path: Path to output MP4 video
        crop_size: Tuple of (width, height) for output crop
        target_fps: Target FPS for output video
        detector_backend: Face detector backend
        smoothing_sigma: Sigma parameter for smoothing function
        buffer_size: Number of frames to buffer for smoothing
        show_preview: Show preview window during processing
    """
    
    # Open video
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {input_video_path}")
    
    # Get video properties
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Set output FPS
    if target_fps is None:
        target_fps = original_fps
    
    print(f"Video info: {frame_width}x{frame_height}, {original_fps:.2f} FPS, {total_frames} frames")
    print(f"Output: {crop_size[0]}x{crop_size[1]}, {target_fps:.2f} FPS")
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(
        output_video_path, 
        fourcc, 
        target_fps, 
        crop_size
    )
    
    # Initialize buffers for smoothing
    centers_x_buffer = []
    centers_y_buffer = []
    frames_buffer = []
    detection_failed_frames = []
    
    # First pass: Collect all face centers
    print("\nFirst pass: Detecting faces and collecting center positions...")
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % 30 == 0:  # Progress update every 30 frames
            print(f"  Processing frame {frame_count}/{total_frames}")
        
        try:
            # Detect face
            face_objs = DeepFace.extract_faces(
                frame, 
                detector_backend=detector_backend, 
                enforce_detection=True
            )
            
            if face_objs and len(face_objs) > 0:
                # Get the first detected face
                facial_area = face_objs[0]['facial_area']
                x, y, w, h = facial_area['x'], facial_area['y'], facial_area['w'], facial_area['h']
                
                # Calculate face center
                center_x = x + w // 2
                center_y = y + h // 2
                
                centers_x_buffer.append(center_x)
                centers_y_buffer.append(center_y)
                frames_buffer.append(frame)
                detection_failed_frames.append(False)
            else:
                # No face detected
                detection_failed_frames.append(True)
                frames_buffer.append(frame)
                
                # Use previous center if available, otherwise None
                if centers_x_buffer:
                    centers_x_buffer.append(centers_x_buffer[-1])
                    centers_y_buffer.append(centers_y_buffer[-1])
                else:
                    centers_x_buffer.append(frame_width // 2)  # Default to frame center
                    centers_y_buffer.append(frame_height // 2)
                    
        except Exception as e:
            # Face detection failed
            print(f"  Warning: Detection failed on frame {frame_count}: {str(e)}...")
            detection_failed_frames.append(True)
            frames_buffer.append(frame)
            
            if centers_x_buffer:
                centers_x_buffer.append(centers_x_buffer[-1])
                centers_y_buffer.append(centers_y_buffer[-1])
            else:
                centers_x_buffer.append(frame_width // 2)
                centers_y_buffer.append(frame_height // 2)
        
        frame_count += 1
    
    print(f"\nCollected {len(centers_x_buffer)} face positions")
    
    # Apply smoothing to center positions
    print("\nApplying smoothing to face center positions...")
    centers_x_smooth, centers_y_smooth = smooth(
        centers_x_buffer, 
        centers_y_buffer, 
        sigma=smoothing_sigma
    )
    
    # Second pass: Create smoothed cropped video
    print("\nSecond pass: Generating smoothed cropped video...")
    
    # Reset video capture to beginning
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    # Calculate frame step for target FPS
    frame_step = max(1, int(original_fps / target_fps))
    
    frame_idx = 0
    output_frame_count = 0
    
    while frame_idx < len(frames_buffer):
        frame = frames_buffer[frame_idx]
        
        # Get smoothed center for this frame
        center_x = centers_x_smooth[frame_idx]
        center_y = centers_y_smooth[frame_idx]
        
        # Calculate crop boundaries
        crop_w, crop_h = crop_size
        half_w, half_h = crop_w // 2, crop_h // 2
        
        # Ensure crop stays within frame boundaries
        x1 = max(0, center_x - half_w)
        y1 = max(0, center_y - half_h)
        x2 = min(frame_width, center_x + half_w)
        y2 = min(frame_height, center_y + half_h)
        
        # Adjust if crop is too close to edges
        if x2 - x1 < crop_w:
            if x1 == 0:
                x2 = min(frame_width, crop_w)
            else:
                x1 = max(0, frame_width - crop_w)
        
        if y2 - y1 < crop_h:
            if y1 == 0:
                y2 = min(frame_height, crop_h)
            else:
                y1 = max(0, frame_height - crop_h)
        
        # Crop the frame
        cropped_frame = frame[y1:y2, x1:x2]
        
        # Resize if necessary
        if cropped_frame.shape[1] != crop_w or cropped_frame.shape[0] != crop_h:
            cropped_frame = cv2.resize(cropped_frame, crop_size)
        
        # Write to output video
        out.write(cropped_frame)
        output_frame_count += 1
        
        # Show preview if requested
        if show_preview and frame_idx % 30 == 0:
            # Create preview frame
            preview = frame.copy()
            
            # Draw original detection window (if detection succeeded)
            if not detection_failed_frames[frame_idx]:
                try:
                    face_objs = DeepFace.extract_faces(
                        frame, 
                        detector_backend=detector_backend, 
                        enforce_detection=False
                    )
                    if face_objs and len(face_objs) > 0:
                        facial_area = face_objs[0]['facial_area']
                        x, y, w, h = facial_area['x'], facial_area['y'], facial_area['w'], facial_area['h']
                        cv2.rectangle(preview, (x, y), (x + w, y + h), (0, 255, 0), 2)
                except:
                    pass
            
            # Draw smoothed crop window
            cv2.rectangle(preview, (x1, y1), (x2, y2), (255, 0, 0), 3)
            cv2.circle(preview, (center_x, center_y), 5, (0, 0, 255), -1)
            
            # Add text info
            cv2.putText(preview, f"Frame: {frame_idx}/{len(frames_buffer)}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            cv2.putText(preview, f"Crop center: ({center_x}, {center_y})", (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
            # Show preview
            cv2.imshow('Smoothed Face Crop Preview', preview)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        # Jump to next frame based on target FPS
        frame_idx += frame_step
    
    # Cleanup
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    
    print(f"\nProcessing complete!")
    print(f"Output video saved to: {output_video_path}")
    print(f"Total frames written: {output_frame_count}")
    print(f"Frames with detection failures: {sum(detection_failed_frames)}")

In [ ]:
from pathlib import Path

data_dir = Path("../untracked")
video_path = data_dir / "vz_20s.mp4"
output_deepface_path = data_dir / "deepface.mp4"

# Two-pass processing with Gaussian smoothing
process_video_with_smoothed_crop(
    input_video_path=str(video_path),
    output_video_path=str(output_deepface_path),
    crop_size=(224, 224),
    target_fps=30,
    detector_backend='opencv',
    smoothing_sigma=2.0,
    show_preview=True
)